## Convert Air Quality Maps from COG to Zarr

In [2]:
import xarray as xr
import rioxarray
import pandas as pd
import os
from pathlib import Path

In [3]:
def tif_lists(root_dir):   
    root_dir = Path(root_dir)
    
    fl = list(map(str, root_dir.rglob("*.tif"))) 
    fl.sort()
    value_tifs =  list(filter(lambda k: '.pse_' not in k, fl))
    error_tifs =  list(filter(lambda k: '.pse_' in k, fl))
    return value_tifs, error_tifs

# 2. Helper to create a dataset from a list of files
def create_dataset(file_list, name):
    ds = xr.open_mfdataset(
        file_list, 
        #chunks=time_series_chunks, 
        combine="nested", 
        concat_dim="time"
    )
    return ds.rename({'band_data': name})

# Chunking strategy:
def make_chunks(ds, delta_t):
    t = ds.dims['time']
    x = ds.dims['x']
    y = ds.dims['y']

    # optimize for daily time series
    if delta_t == "D":       
        ch = {'time': t, 'x': 20, 'y': 20}

    # optimize for single time step maps
    else:
        ch = {'time': 1, 'x': x, 'y': y}
    return ch

# lookup dict for description attributes
temp_agg_dict = {
    'O3_max': 'daily max of 8h rolling mean', 
    'O3_perc': '93.14th percentile of daily max of 8h rolling mean', 
    'NO2_mean': 'mean', 
    'PM10_mean': 'mean',
    'PM10_perc': '91.4th percentile of daily mean',
    'PM2.5_mean':  'mean', 
}

# lookup dict for file naming
temp_step_dict = {
    'D': 'daily',
    'M': 'monthly',
    'Y': 'annual',
}

# complete conversion function
def write_zarr(var, t_start="2015-01-01", delta_t = "D", res = "10km"):
    
    name = var.split("_")[0]
    temp = temp_step_dict.get(delta_t)
    
    var_dir = Path('/palma/scratch/tmp/jheisig/aq/AQ_maps/v20250306/', temp, var, 'cog')
    out_dir = Path('/palma/scratch/tmp/jheisig/aq/AQ_maps/v20250306/zarr_stores', temp)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    value_tifs, error_tifs = tif_lists(var_dir)
    
    ending = "_".join(Path(value_tifs[-1]).stem.split("_")[-4:-1])
    out_file = f"{var.lower()}_{temp_step_dict.get(delta_t)}_{res}_s_{t_start.replace('-','')}_{ending}_v1.zarr"
    out_zarr = Path(out_dir, out_file)

    if out_zarr.is_dir():
        print(var, ": completed.")
        
    else:
        print(out_file)

        ds_values = create_dataset(value_tifs, "value").squeeze("band", drop=True)
        ds_errors = create_dataset(error_tifs, "error").squeeze("band", drop=True)

        # Merge them into one object
        full_dataset = xr.merge([ds_values, ds_errors])

        # Add metadata so you know it's PM2.5
        full_dataset.attrs["variable"] = name
        full_dataset.value.attrs["long_name"] = f"{name} concentration"
        full_dataset.error.attrs["long_name"] = f"{name} prediction error"
        full_dataset.attrs["units"] = "ug/m3"
        full_dataset.attrs["temporal aggregation"] = temp_agg_dict.get(var)

        # date range
        dates = pd.date_range(start=t_start, periods=len(full_dataset.time), freq=delta_t)
        full_dataset = full_dataset.assign_coords(time=dates)
        
        # chunks
        full_dataset = full_dataset.chunk(chunks=make_chunks(full_dataset, delta_t))
        
        full_dataset.to_zarr(out_zarr, mode='w', compute=True)


### Daily maps

In [4]:
daily_home = '/palma/scratch/tmp/jheisig/aq/AQ_maps/v20250306/daily'
daily_dirs = os.listdir(daily_home)
daily_dirs

['O3_max', 'NO2_mean', 'PM10_mean', 'PM2.5_mean']

In [5]:
for d in daily_dirs:
    t0 = "2015-01-01"
    if d == "NO2_mean":
        t0 = "2018-05-01"
        
    write_zarr(d, t_start=t0)    


O3_max : completed.
NO2_mean : completed.
PM10_mean : completed.
PM2.5_mean : completed.


### Monthly maps

In [6]:
monthly_dirs = os.listdir('/palma/scratch/tmp/jheisig/aq/AQ_maps/v20250306/monthly')
monthly_dirs

['NO2_mean', 'PM10_mean', 'O3_perc', 'PM10_perc', 'PM2.5_mean']

In [7]:
for d in monthly_dirs:
    t0 = "2015-01-01"
    if d == "NO2_mean":
        t0 = "2018-05-01"
        
    write_zarr(d, t_start=t0, delta_t = "M", res = '1km')    


NO2_mean : completed.
PM10_mean : completed.
O3_perc : completed.
PM10_perc : completed.
pm2.5_mean_monthly_1km_s_20150101_20231231_eu_epsg.3035_v1.zarr


### Annual maps

In [8]:
annual_dirs = os.listdir('/palma/scratch/tmp/jheisig/aq/AQ_maps/v20250306/annual')
annual_dirs

['NO2_mean', 'PM10_mean', 'O3_perc', 'PM10_perc', 'PM2.5_mean']

In [9]:
for d in annual_dirs:
    t0 = "2015-01-01"
    if d == "NO2_mean":
        t0 = "2018-05-01"
        
    write_zarr(d, t_start=t0, delta_t = "Y", res = '1km')    


NO2_mean : completed.
PM10_mean : completed.
O3_perc : completed.
PM10_perc : completed.
PM2.5_mean : completed.


Check if chunking was applied correctly:

In [10]:
z = xr.open_dataset("/palma/scratch/tmp/jheisig/aq/AQ_maps/v20250306/zarr_stores/daily/no2_mean_daily_10km_s_20180501_20231231_eu_epsg.3035_v1.zarr", engine="zarr")
z.value.encoding

{'chunks': (2066, 20, 20),
 'preferred_chunks': {'time': 2066, 'y': 20, 'x': 20},
 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0),
 'filters': None,
 '_FillValue': nan,
 'scale_factor': 1.0,
 'add_offset': 0.0,
 'dtype': dtype('float32')}

In [12]:
z = xr.open_dataset("/palma/scratch/tmp/jheisig/aq/AQ_maps/v20250306/zarr_stores/annual/o3_perc_annual_1km_s_20150101_20231231_eu_epsg.3035_v1.zarr", engine="zarr")
z.value.encoding

{'chunks': (1, 4100, 4900),
 'preferred_chunks': {'time': 1, 'y': 4100, 'x': 4900},
 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0),
 'filters': None,
 '_FillValue': nan,
 'scale_factor': 1.0,
 'add_offset': 0.0,
 'dtype': dtype('float32')}